In [1]:
import qiskit
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

print("Qiskit Version:", qiskit.__version__)
simulator = AerSimulator()
print(simulator)

def build_deutsch_oracle(function_type):
    """Build the oracle U_f for one of the four possible Boolean functions."""
    qc = QuantumCircuit(2, name=f"Oracle_{function_type}")

    if function_type == "constant_0":
        # f(x) = 0: do nothing
        pass

    elif function_type == "constant_1":
        # f(x) = 1: flip the target qubit
        qc.x(1)

    elif function_type == "balanced_x":
        # f(x) = x: target ^= x
        qc.cx(0, 1)

    elif function_type == "balanced_not_x":
        # f(x) = 1 - x: target ^= (1 - x)
        qc.x(1)
        qc.cx(0, 1)

    else:
        raise ValueError("Unknown function type.")

    return qc


def deutsch_algorithm(function_type, shots=1024):
    """Run Deutsch's algorithm and return the circuit and measurement counts."""
    oracle = build_deutsch_oracle(function_type)

    qc = QuantumCircuit(2, 1)

    # Prepare |0>|1>
    qc.x(1)

    # Create superposition
    qc.h(0)
    qc.h(1)

    # One oracle query
    qc.compose(oracle, inplace=True)

    # Interference on the control qubit
    qc.h(0)

    # Measure only the control qubit
    qc.measure(0, 0)

    counts = simulator.run(qc, shots=shots).result().get_counts()
    return qc, counts


def classify_deutsch_result(counts):
    """Interpret the control-qubit measurement."""
    result = max(counts, key=counts.get)
    return "constant" if result == "0" else "balanced"


Qiskit Version: 2.5.1
AerSimulator('aer_simulator')


In [2]:
print("""Conceptual binary-classification example:

Input qubit:
    |x>  -> represents a binary feature or decision input

Oracle:
    U_f |x>|->  ->  (-1)^f(x) |x>|->

Phase kickback:
    The classification value f(x) is encoded as a phase on the
    input/control state rather than being directly measured.

Interference:
    A later Hadamard gate converts the phase information into
    a measurable 0/1 outcome.

Interpretation:
    0 -> one class / decision category
    1 -> the other class / decision category

This is a conceptual analogy. Deutsch's algorithm itself solves
a specific Boolean-function problem rather than a general machine-
learning classification task.
""")


Conceptual binary-classification example:

Input qubit:
    |x>  -> represents a binary feature or decision input

Oracle:
    U_f |x>|->  ->  (-1)^f(x) |x>|->

Phase kickback:
    The classification value f(x) is encoded as a phase on the
    input/control state rather than being directly measured.

Interference:
    A later Hadamard gate converts the phase information into
    a measurable 0/1 outcome.

Interpretation:
    0 -> one class / decision category
    1 -> the other class / decision category

This is a conceptual analogy. Deutsch's algorithm itself solves
a specific Boolean-function problem rather than a general machine-
learning classification task.

